# Continuous tuning on GEAP: SFT → RLFT

This notebook is a **thin demo**: every step calls into the tested
`geap_tuning` package. See
[`docs/notes/checkpoints-and-continuous-tuning.md`](../docs/notes/checkpoints-and-continuous-tuning.md)
for the verified SDK surface.

The documented GEAP best practice is **SFT first, then continuous-tune**: teach
the skill/format supervised, then refine with RL. Here we chain
[SFT](01_sft.ipynb) → [RLFT](03_rlft.ipynb) on the same math domain. Stage 2's
`base_model` is stage 1's **tuned-model resource name**
(`projects/.../models/id@ver`); the Gen AI SDK auto-detects the `projects/`
prefix and treats it as a pre-tuned model — no extra plumbing.

> **Constraints:** continuous tuning is **Gen AI SDK / Vertex only**; the base
> SFT model must have been tuned on/after 2025-07-11; both stages run in the
> **same region**; tuning stays regional (never `global`).

> **Requires live GCP and incurs tuning cost (two jobs).** Have a real `.env`
> and `gcloud auth` in place before running the tune/eval cells.

In [ ]:
from geap_tuning.config import genai_client, load_config

# Both stages tune gemini-3.5-flash in the SAME region. VERSION parameterizes the
# display names so reruns reuse the same jobs (cost control).
BASE_MODEL = "gemini-3.5-flash"
VERSION = "v1"
cfg = load_config()
client = genai_client(cfg)
cfg

## Stage 1 — SFT seed

RLFT records carry no gold completion, so the SFT stage uses
`build_math_sft_dataset` — the *same* `MATH_PROBLEMS`, but with a gold
`Answer: <n>` model turn — to teach the answer **format**. RLFT then refines
**correctness** on top.

In [ ]:
from geap_tuning.gcs import upload_file
from geap_tuning.jobs import find_tuning_job_by_display_name, tuned_model_name, wait_for_tuning_job
from geap_tuning.rlft.data import build_math_sft_dataset
from geap_tuning.sft.tune import launch_sft_job

sft_paths = build_math_sft_dataset("../datasets/math_sft")
sft_train = upload_file(sft_paths["train"], f"{cfg.bucket}/cont_sft/train.jsonl")
sft_val = upload_file(sft_paths["val"], f"{cfg.bucket}/cont_sft/val.jsonl")

SFT_NAME = f"geap-cont-sft-{VERSION}"
sft_job = find_tuning_job_by_display_name(client, SFT_NAME)
if sft_job is None:
    sft_job = launch_sft_job(
        client,
        train_uri=sft_train,
        val_uri=sft_val,
        display_name=SFT_NAME,
        base_model=BASE_MODEL,
    )
sft_job = wait_for_tuning_job(client, sft_job.name)
sft_model = tuned_model_name(sft_job)  # projects/.../models/id@ver
sft_model

## Stage 2 — RLFT continued from the SFT model

Pass `sft_model` (the stage-1 resource name) as `base_model`. Preflight the
reward first, exactly as in the RLFT notebook.

In [ ]:
from geap_tuning.rlft.data import (
    MATH_PROBLEMS,
    build_rlft_dataset,
    build_rlft_records,
    split_dataset,
)
from geap_tuning.rlft.tune import launch_rlft_job, validate_reward_config

rlft_paths = build_rlft_dataset("../datasets/rlft_math")
train_records = build_rlft_records(split_dataset(MATH_PROBLEMS)[0])
validate_reward_config(
    client,
    project=cfg.project,
    location=cfg.location,
    sample_answer="Answer: 4",
    example_record=train_records[0],
)

rlft_train = upload_file(rlft_paths["train"], f"{cfg.bucket}/cont_rlft/train.jsonl")
rlft_val = upload_file(rlft_paths["val"], f"{cfg.bucket}/cont_rlft/val.jsonl")

RLFT_NAME = f"geap-cont-rlft-{VERSION}"
rlft_job = find_tuning_job_by_display_name(client, RLFT_NAME)
if rlft_job is None:
    rlft_job = launch_rlft_job(
        client,
        train_uri=rlft_train,
        val_uri=rlft_val,
        display_name=RLFT_NAME,
        base_model=sft_model,  # continue-tune from the SFT model
    )
rlft_job = wait_for_tuning_job(client, rlft_job.name)
rlft_job.state

## Compare the lift

Held-out math accuracy for the SFT-seed model vs. the continuously-tuned RLFT
model, scored by the same reward the RLFT notebook uses.

In [ ]:
from geap_tuning.inference import generate
from geap_tuning.jobs import tuned_endpoint
from geap_tuning.rlft.evaluate import run_rlft_eval

_, _, test_problems = split_dataset(MATH_PROBLEMS)
test_records = build_rlft_records(test_problems)

sft_endpoint = tuned_endpoint(sft_job)
rlft_endpoint = tuned_endpoint(rlft_job)
sft_metrics = run_rlft_eval(test_records, lambda t: generate(client, sft_endpoint, t))
rlft_metrics = run_rlft_eval(test_records, lambda t: generate(client, rlft_endpoint, t))
print(f"SFT seed accuracy:        {sft_metrics['accuracy']:.3f} (n={sft_metrics['n']})")
print(f"Continuous RLFT accuracy: {rlft_metrics['accuracy']:.3f} (n={rlft_metrics['n']})")

## Next steps

The same mechanic covers the other supported chains — **SFT→SFT**, **SFT→DPO**,
and **RLFT→RLFT** — just point stage 2's `base_model` at the prior tuned model
and pick a source checkpoint with `pre_tuned_model_checkpoint_id`. Pair this with
[checkpointing](04_checkpoints.ipynb) to continue from a specific epoch.